# Kimi-Linear (GDN-2) Code LM — train on **Google Colab (T4)**

Trains the hybrid Gated-DeltaNet-2 / MLA / MoE code language model on **CodeParrot**
using the `configs/colab_t4.yaml` recipe (~190M params, bfloat16, single GPU).

**Before you start:** `Runtime -> Change runtime type -> Hardware accelerator = T4 GPU`.

The pipeline: install -> get code -> prepare data (tokenize) -> train -> evaluate -> generate.


## 0. Confirm the GPU

In [1]:
!nvidia-smi

Thu Jul 16 04:11:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    print("Not running on Google Colab, skipping drive mount.")

Mounted at /content/drive


In [3]:
try:
    from huggingface_hub import notebook_login
    notebook_login()
except:
    print("Cannot login to Hugging Face Hub.")

## 1. Get the project

Set `REPO_URL` to your repository, or upload the project folder to `/content` and
skip the clone. Everything below assumes we `cd` into the project root.


In [4]:
import os

REPO_URL = "https://github.com/wisnunugroho21/nugie-codeparrot.git"  # <-- EDIT ME
PROJECT_DIR = "/content/nugie-codeparrot"

if not os.path.isdir(PROJECT_DIR):
    !git clone $REPO_URL $PROJECT_DIR
%cd $PROJECT_DIR
!ls


Cloning into '/content/nugie-codeparrot'...
remote: Enumerating objects: 197, done.
remote: Counting objects: 100% (197/197), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 197 (delta 108), reused 138 (delta 59), pack-reused 0 (from 0)
Receiving objects: 100% (197/197), 158.52 KiB | 4.95 MiB/s, done.
Resolving deltas: 100% (108/108), done.
/content/nugie-codeparrot
configs		     multi_latent_attention  pyproject.toml    tests
gated_deltanet_2     notebooks		     README.md
kimi_linear_gdn2.py  pipeline		     requirements.txt


In [1]:
%cd /content/nugie-codeparrot
!git checkout muon_optimizer
!git pull

/content/nugie-codeparrot
Already on 'muon_optimizer'
Your branch is up to date with 'origin/muon_optimizer'.
Already up to date.


## 2. Install dependencies

We install the project's requirements first, then overlay the **CUDA 12 build of JAX**
last so the GPU plugin wins.


In [14]:
!pip install -q -U -r requirements.txt
!pip install -q -U "jax[cuda12]"


## 3. (Optional) Hugging Face token

CodeParrot streams fine anonymously, but a token raises the download rate limit. Skip
this cell if you don't have one.


In [ ]:
# from huggingface_hub import login
# login()  # paste your HF token when prompted


## 4. Verify JAX sees the T4

In [2]:
import jax
print("JAX", jax.__version__, "devices:", jax.devices())
assert any(d.platform == "gpu" for d in jax.devices()), "No GPU visible — set the runtime to T4."


JAX 0.10.2 devices: [CudaDevice(id=0)]


## 5. Build the config

We load `configs/colab_t4.yaml` and apply **demo-friendly overrides** (small corpus,
few steps) so this notebook finishes quickly. Delete the overrides for a real run.
We reuse this one `cfg` object for every stage below.


In [3]:
from pipeline.config import ExperimentConfig

cfg = ExperimentConfig.load("configs/colab_t4.yaml")

# --- Demo overrides (comment out for a full training run) ---
cfg.data.num_train_docs = 50000      # full config: 50000
cfg.data.num_val_docs   = 200
cfg.train.max_steps     = 40000      # full config: 40000
cfg.train.warmup_steps  = 100
cfg.train.eval_every    = 500
cfg.train.save_every    = 50
cfg.train.log_every     = 25
cfg.train.batch_size    = 8

# --- Optional: persist checkpoints to Google Drive (Colab runtimes are ephemeral) ---
from google.colab import drive;
drive.mount("/content/drive")
cfg.train.ckpt_dir = "/content/drive/MyDrive/nugie-codeparrot/checkpoints/colab_t4"

cfg.validate()
print("compute_dtype:", cfg.model.compute_dtype, "| seq_len:", cfg.data.seq_len,
      "| batch:", cfg.train.batch_size, "| max_steps:", cfg.train.max_steps)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
compute_dtype: bfloat16 | seq_len: 512 | batch: 8 | max_steps: 40000


## 6. Prepare the data

Streams `codeparrot/codeparrot-train-v2-near-dedup` from the Hub, tokenizes with the CodeParrot BPE
tokenizer, and writes packed `train.bin` / `val.bin` memmaps + `meta.json`. Runs once.


In [4]:
from pipeline import prepare_data
prepare_data.prepare(cfg)


config.json:   0%|          | 0.00/927 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 32767), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 32767), got 50256. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/497k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/277k [00:00<?, ?B/s]

...

[huggingface] codeparrot/codeparrot-train-v2-near-dedup tok=codeparrot vocab=32768 dtype=uint16


Resolving data files:   0%|          | 0/52 [00:00<?, ?it/s]

Tokenizing validation split...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2605 > 1024). Running this sequence through the model will result in indexing errors


Tokenizing training split...
  tokenized 1000 docs...
  tokenized 2000 docs...
  tokenized 3000 docs...
  tokenized 4000 docs...
  tokenized 5000 docs...
  tokenized 6000 docs...
  tokenized 7000 docs...
  tokenized 8000 docs...
  tokenized 9000 docs...
  tokenized 10000 docs...
  tokenized 11000 docs...
  tokenized 12000 docs...
  tokenized 13000 docs...
  tokenized 14000 docs...
  tokenized 15000 docs...
  tokenized 16000 docs...
  tokenized 17000 docs...
  tokenized 18000 docs...
  tokenized 19000 docs...
  tokenized 20000 docs...
  tokenized 21000 docs...
  tokenized 22000 docs...
  tokenized 23000 docs...
  tokenized 24000 docs...
  tokenized 25000 docs...
  tokenized 26000 docs...
  tokenized 27000 docs...
  tokenized 28000 docs...
  tokenized 29000 docs...
  tokenized 30000 docs...
  tokenized 31000 docs...
  tokenized 32000 docs...
  tokenized 33000 docs...
  tokenized 34000 docs...
  tokenized 35000 docs...
  tokenized 36000 docs...
  tokenized 37000 docs...
  tokenized 38000 

## 7. Train

Streams metrics (loss / perplexity / aux / lr / tokens-per-sec) and writes Orbax
checkpoints to `cfg.train.ckpt_dir`. Re-run with `train.train(cfg, resume=True)` to
continue from the latest checkpoint.


In [6]:
from pipeline import train
# train.train(cfg)
train.train(cfg, resume=True)

JAX devices: [CudaDevice(id=0)]
optimizer: Muon on 67,280,896 2D params, AdamW on 125,914,176 others (1D biases/norms/decays, 3D experts/conv)
Model params: 193,195,072  (compute_dtype=bfloat16, seq_len=512)


JaxRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 5.99GiB. [tf-allocator-allocation-error=''] [executable_name='jit_add']

## 8. Evaluate

Restores the latest checkpoint and reports validation cross-entropy / perplexity /
bits-per-token over a capped number of batches.


In [ ]:
from pipeline import evaluate
evaluate.run_eval(cfg, step=None, max_batches=50)


## 9. Generate code

Autoregressive completion via the model's streaming decode. NOTE: after only the short
demo run above the output will be weak — train for many more steps for good code.


In [ ]:
evaluate.run_generate(cfg, step=None, prompt="def fibonacci(n):\n", max_new_tokens=128)
